Mock Sales Order Data Generator

~7,000 orders · ~17,500 items | Coverage: 2023-01-01 → 2026-06-30 (42 months)

Targets:

    h2b_bdc_salesorder.salesorder.salesorder
    h2b_bdc_salesorder.salesorder.sales_order_item

Depends on :

    h2b_bdc_product.product.product
    h2b_bdc_customer.customer.customersalesarea
    h2b_bdc_customer.customer.customer
    h2b_bdc_weather.weather.weather 

In [ ]:
# Create the catalog
spark.sql("CREATE CATALOG IF NOT EXISTS h2b_dbx_salesorder")


In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

schemas = [s.name for s in w.schemas.list(catalog_name="h2b_bdc_salesorder") if s.name != "information_schema"]

tables = [
    (table.schema_name, table.name)
    for schema in schemas
    for table in w.tables.list(catalog_name="h2b_bdc_salesorder", schema_name=schema)
]

In [ ]:
# forced clone

for schema, table in tables:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS h2b_dbx_salesorder.{schema}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS h2b_dbx_salesorder.{schema}.{table}
        AS SELECT * FROM h2b_bdc_salesorder.{schema}.{table}
    """)

In [ ]:
for schema, table in tables:
    spark.sql(f"TRUNCATE TABLE h2b_dbx_salesorder.{schema}.{table}")

In [ ]:


import numpy as np
import pandas as pd
from datetime import timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
     


In [ ]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_CUSTOMERS = [
    '10100006', '10100002', '12200001',
    '10100012', '10186001', '10186002', '10186003',
    'EWM10-CU01', 'EWM10-CU02', 'EWM10-CU03',
]

_TIER_MAP = {
    '10100006': 'A', '10100002': 'A', '12200001': 'A',
    '10100012': 'B', '10186001': 'B', '10186002': 'B', '10186003': 'B',
    'EWM10-CU01': 'C', 'EWM10-CU02': 'C', 'EWM10-CU03': 'C',
}
_TIER_WEIGHT = {'A': 6, 'B': 2, 'C': 1}

_MATERIALS = ['TG11', 'TG12', 'FPP', 'RTE', 'CM-FL-V00', 'CM-MLFL-KM-VXX']

_MATERIAL_GROUP = {
    'TG11': 'L001', 'TG12': 'L001',
    'FPP':  'P001', 'RTE':  'P001',
    'CM-FL-V00': 'L004', 'CM-MLFL-KM-VXX': 'L004',
}
_PRICE_RANGES = {
    'TG11': (8.0, 18.0),  'TG12': (8.0, 18.0),
    'FPP':  (5.0, 12.0),  'RTE':  (5.0, 12.0),
    'CM-FL-V00':      (15.0, 35.0),
    'CM-MLFL-KM-VXX': (15.0, 35.0),
}
_PHASE_OUT = pd.Timestamp('2025-06-01')

_TARGET_SO  = 'h2b_dbx_salesorder.salesorder.salesorder'
_TARGET_SOI = 'h2b_dbx_salesorder.salesorder.salesorderitem'
_N_ORDERS   = 7000

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _seasonal_offset(month: int) -> float:
    if month in (7, 8):   return  0.40
    if month in (10, 11): return  0.20
    if month in (1, 2):   return -0.20
    return 0.0


def _seasonal_weight(month: int, year: int, material_group: str) -> float:
    """Seasonal weight * YoY growth factor per material group."""
    offset = _seasonal_offset(month)
    growth = 1.0 + 0.03 * (year - 2023)
    if material_group == 'L001':
        sw = 1.0 + offset
    elif material_group == 'L004':
        sw = 1.0 + offset * 0.50
    else:  # P001
        sw = 1.0 + offset * 0.05
    return sw * growth


def _weather_mult(avg_t: float, anom: float, material_group: str) -> float:
    """Temperature-driven demand multiplier. Hard cap at 3×."""
    mult = 1.0
    if avg_t > 22:
        mult += 0.50 * min((avg_t - 22) / 5.0, 1.0)  # ramp +0%→+50% from 22°C to 27°C
    if avg_t > 25:
        mult += 0.30                                   # extra boost above 25°C
    mult += 0.20 * max(0.0, anom)                      # positive anomaly bonus only
    mult = min(mult, 3.0)
    if material_group == 'P001':
        mult = 1.0 + (mult - 1.0) * 0.30              # powder less temp-sensitive
    return mult


def _effective_price(
    material: str, month: int,
    base_prices: dict, surge_skus: set, surge_pct: dict,
) -> float:
    """Apply seasonal price surges to the fixed base price."""
    price = base_prices[material]
    grp   = _MATERIAL_GROUP[material]
    # Jul–Aug: +5% for all non-P001
    if month in (7, 8) and grp != 'P001':
        price *= 1.05
    # Sep–Dec: +5–10% for the 2 designated surge SKUs
    if month in (9, 10, 11, 12) and material in surge_skus:
        price *= (1.0 + surge_pct[material])
    return round(price, 2)

In [ ]:
# ---------------------------------------------------------------------------
# Generator
# ---------------------------------------------------------------------------

def generate_sales_order_tables(
    reference_date: str = '2023-01-01',
    months: int = 42,
) -> dict:
    """
    Generate synthetic SalesOrder + SalesOrderItem tables.

    Parameters
    ----------
    reference_date : str   First month of the training window (YYYY-MM-DD).
    months         : int   Number of months to cover (default 42 = 36 training + 6 test).

    Returns
    -------
    dict  {'SalesOrder': df1, 'SalesOrderItem': df2}
    """
    spark = SparkSession.builder.getOrCreate()

    # ── Fixed draws — seed=42 ───────────────────────────────────────────────
    rng = np.random.default_rng(seed=42)

    # Base prices: 6 draws (one per material, in _MATERIALS order)
    base_prices = {mat: round(float(rng.uniform(*_PRICE_RANGES[mat])), 2)
                   for mat in _MATERIALS}

    # Sep-Dec price surge: 1 random P001 SKU + CM-MLFL-KM-VXX fixed
    p001_surge = str(rng.choice(['FPP', 'RTE']))          # 1 draw
    surge_skus = {p001_surge, 'CM-MLFL-KM-VXX'}
    surge_pct  = {
        p001_surge:        round(float(rng.uniform(0.05, 0.10)), 4),  # 1 draw
        'CM-MLFL-KM-VXX':  round(float(rng.uniform(0.05, 0.10)), 4), # 1 draw
    }



    # ── Read real data ──────────────────────────────────────────────────────
    products_pdf = (
        spark.table('h2b_bdc_product.product.product')
        .filter(F.col('Product').isin(_MATERIALS))
        .select('Product', 'ProductGroup', 'ProductHierarchy',
                'CrossPlantStatus', 'ProductExternalID')
        .toPandas()
        .set_index('Product')
    )

    ca_raw = (
        spark.table('h2b_bdc_customer.customer.customersalesarea')
        .filter(F.col('Customer').isin(_CUSTOMERS))
        .select('Customer', 'DistributionChannel', 'CustomerPaymentTerms',
                'IncotermsClassification', 'Currency', 'CustomerGroup')
        .toPandas()
    )
    # Prefer DistributionChannel='10'; fall back to any available row
    ca_10    = ca_raw[ca_raw['DistributionChannel'] == '10'].drop_duplicates('Customer')
    ca_other = ca_raw[~ca_raw['Customer'].isin(ca_10['Customer'])].drop_duplicates('Customer')
    customer_area_pdf = pd.concat([ca_10, ca_other]).set_index('Customer')

    customer_geo_pdf = (
        spark.table('h2b_bdc_customer.customer.customer')
        .filter(F.col('Customer').isin(_CUSTOMERS))
        .select('Customer', 'Country', 'Region')
        .toPandas()
        .drop_duplicates(subset='Customer')
        .set_index('Customer')
    )

    weather_pdf = spark.table('h2b_bdc_weather.weather.weather').toPandas()
    weather_lookup = {
        (r.country, r.region, int(r.year), int(r.month)):
            (float(r.avg_temp_c), float(r.temp_anomaly_c))
        for r in weather_pdf.itertuples()
    }

    # ── Month-level order allocation ────────────────────────────────────────
    month_starts = pd.date_range(start=reference_date, periods=months, freq='MS')

    month_weights = np.array([
        max(1.0 + _seasonal_offset(ts.month) * (1.0 + 0.03 * (ts.year - 2023)), 0.01)
        for ts in month_starts
    ])
    month_weights /= month_weights.sum()

    orders_per_month = np.round(month_weights * _N_ORDERS).astype(int)
    # Fix rounding so total == _N_ORDERS exactly
    orders_per_month[month_weights.argmax()] += _N_ORDERS - orders_per_month.sum()

    # ── Customer sampling weights ───────────────────────────────────────────
    cust_w = np.array([_TIER_WEIGHT[_TIER_MAP[c]] for c in _CUSTOMERS], dtype=float)
    cust_w /= cust_w.sum()

    # ── Generate SalesOrder rows ────────────────────────────────────────────
    so_rows   = []
    so_counter = 1

    for mi, ts in enumerate(month_starts):
        n_month       = int(orders_per_month[mi])
        year, month   = int(ts.year), int(ts.month)
        days_in_month = int((ts + pd.DateOffset(months=1) - ts).days)

        # Pre-compute evenly-spaced day slots with small jitter
        if n_month > 0:
            spacing    = days_in_month / n_month
            centers    = (np.arange(n_month) + 0.5) * spacing
            jitter     = rng.normal(0, max(spacing * 0.25, 0.5), n_month)
            month_days = np.clip(np.round(centers + jitter).astype(int), 1, days_in_month)
            rng.shuffle(month_days)

        for oi in range(n_month):
            customer   = str(rng.choice(_CUSTOMERS, p=cust_w))
            day        = int(month_days[oi])
            creation   = ts.date().replace(day=day)
            so_type    = 'OR' if rng.random() < 0.90 else 'RE'
            deliv_days = int(rng.integers(5, 22))
            delivery   = creation + timedelta(days=deliv_days)
            deliv_blk  = 'ZL' if rng.random() < 0.03 else ''
            bill_blk   = 'Z1' if rng.random() < 0.02 else ''
            sd_r       = rng.random()
            sd_status  = 'A' if sd_r < 0.30 else ('C' if sd_r > 0.80 else 'B')
            rjcn_r     = rng.random()
            rjcn_sts   = 'C' if rjcn_r < 0.10 else ''
            pay_r      = rng.random()
            pay_method = 'T' if pay_r < 0.70 else ('C' if pay_r > 0.90 else 'D')

            ca        = customer_area_pdf.loc[customer] if customer in customer_area_pdf.index else None
            dist_ch   = str(ca['DistributionChannel'])      if ca is not None else '10'
            currency  = str(ca['Currency'])                 if ca is not None else 'EUR'
            pay_terms = str(ca['CustomerPaymentTerms'])     if ca is not None else 'NT30'
            incoterms = str(ca['IncotermsClassification'])  if ca is not None else 'EXW'
            cust_grp  = str(ca['CustomerGroup'])            if ca is not None else 'KG01'

            so_rows.append({
                'SalesOrder':                    f'SO-{so_counter:06d}',
                'SalesOrderType':                so_type,
                'CreationDate':                  creation,
                'SalesOrderDate':                creation,
                'SoldToParty':                   customer,
                'SalesOrganization':             'VKORG1',
                'DistributionChannel':           dist_ch,
                'OrganizationDivision':          '01',
                'SalesGroup':                    'SG01',
                'SalesOffice':                   'SO01',
                'TransactionCurrency':           currency,
                'TotalNetAmount':                0.0,      # back-filled after items
                'CustomerPaymentTerms':          pay_terms,
                'RequestedDeliveryDate':         delivery,
                'IncotermsClassification':       incoterms,
                'DeliveryBlockReason':           deliv_blk,
                'HeaderBillingBlockReason':      bill_blk,
                'OverallSDProcessStatus':        sd_status,
                'OverallSDDocumentRejectionSts': rjcn_sts,
                'FiscalYear':                    str(year),
                'FiscalPeriod':                  str(month).zfill(2),
                'BillingCompanyCode':            'CC01',
                'PaymentMethod':                 pay_method,
                'AdditionalCustomerGroup1':      cust_grp,
            })
            so_counter += 1

    so_pdf = pd.DataFrame(so_rows)

    # ── Generate SalesOrderItem rows ────────────────────────────────────────
    soi_rows = []

    for _, so in so_pdf.iterrows():
        so_id    = so['SalesOrder']
        customer = so['SoldToParty']
        tier     = _TIER_MAP[customer]
        creation = so['CreationDate']
        year     = creation.year
        month    = creation.month
        dist_ch  = so['DistributionChannel']
        currency = so['TransactionCurrency']
        rjcn_sts = so['OverallSDDocumentRejectionSts']

        # Customer geo → weather lookup
        if customer in customer_geo_pdf.index:
            country = customer_geo_pdf.loc[customer, 'Country']
            region  = customer_geo_pdf.loc[customer, 'Region']
        else:
            country = region = None
        avg_t, anom = weather_lookup.get((country, region, year, month), (10.0, 0.0))

        # Per-material sampling weights (seasonal + phase-out)
        mat_w = []
        for mat in _MATERIALS:
            if mat == 'CM-MLFL-KM-VXX' and pd.Timestamp(creation) >= _PHASE_OUT:
                mat_w.append(0.0)
                continue
            grp = _MATERIAL_GROUP[mat]
            w   = 1.0
            if grp == 'L001' and month in (7, 8): w += 0.40
            if grp == 'L004' and month in (7, 8): w += 0.20
            mat_w.append(w)

        valid_mats = [m for m, w in zip(_MATERIALS, mat_w) if w > 0.0]
        valid_w    = np.array([w for w in mat_w if w > 0.0], dtype=float)
        valid_w   /= valid_w.sum()

        n_items = min(int(rng.integers(2, 4)), len(valid_mats))  # 2 or 3
        chosen  = [
            valid_mats[i]
            for i in rng.choice(len(valid_mats), size=n_items, replace=False, p=valid_w)
        ]

        for seq, material in enumerate(chosen):
            mat_group = _MATERIAL_GROUP[material]
            item_no   = f'{(seq + 1) * 10:06d}'

            # Step 1 — base quantity
            if tier == 'A':   base = rng.uniform(400.0, 600.0)
            elif tier == 'B': base = rng.uniform(40.0,  180.0)
            else:             base = rng.uniform(2.0,    15.0)

            # Steps 2–4 — seasonal × weather × lognormal noise
            sw  = _seasonal_weight(month, year, mat_group)
            wm  = _weather_mult(avg_t, anom, mat_group)
            ln  = float(np.exp(rng.normal(0.0, 0.05)))
            qty = max(1, round(base * sw * wm * ln))

            req_qty   = max(qty + 1, round(qty * float(rng.uniform(1.05, 1.15))))
            net_price = _effective_price(material, month, base_prices, surge_skus, surge_pct)
            net_amt   = round(qty * net_price, 2)
            tax_amt   = round(net_amt * 0.19, 2) if currency == 'EUR' else 0.0

            item_cat = 'TAN' if rng.random() < 0.95 else 'TANN'
            rjcn_rsn = 'Z1' if rjcn_sts == 'C' else ''
            del_r    = rng.random()
            del_sts  = 'A' if del_r < 0.30 else ('C' if del_r > 0.70 else 'B')
            bill_blk = 'B' if rng.random() < 0.05 else ''
            del_prio = {'A': '1', 'B': '5', 'C': '9'}[tier]

            pr          = products_pdf.loc[material] if material in products_pdf.index else None
            mat_grp_col = str(pr['ProductGroup'])      if pr is not None else mat_group
            prod_hier   = str(pr['ProductHierarchy'])  if pr is not None else ''
            ean         = str(pr['ProductExternalID']) if pr is not None else ''
            plant       = 'PLANT1' if dist_ch == '10' else 'PLANT2'

            soi_rows.append({
                'SalesOrder':                 so_id,
                'SalesOrderItem':             item_no,
                'Material':                   material,
                'Product':                    material,
                'Plant':                      plant,
                'MaterialGroup':              mat_grp_col,
                'ProductHierarchyNode':       prod_hier,
                'OrderQuantity':              int(qty),
                'OrderQuantityUnit':          'CS',
                'RequestedQuantity':          int(req_qty),
                'RequestedQuantityUnit':      'CS',
                'NetPriceAmount':             net_price,
                'NetAmount':                  net_amt,
                'TaxAmount':                  tax_amt,
                'SalesOrderItemCategory':     item_cat,
                'SalesDocumentRjcnReason':    rjcn_rsn,
                'DeliveryStatus':             del_sts,
                'BillingBlockStatus':         bill_blk,
                'DeliveryPriority':           del_prio,
                'InternationalArticleNumber': ean,
                'SalesOrganization':          'VKORG1',
                'DistributionChannel':        str(dist_ch),
                'SoldToParty':                customer,
                'PayerParty':                 customer,
            })

    soi_pdf = pd.DataFrame(soi_rows)

    # ── Back-fill TotalNetAmount ────────────────────────────────────────────
    total_net = soi_pdf.groupby('SalesOrder')['NetAmount'].sum().round(2)
    so_pdf['TotalNetAmount'] = so_pdf['SalesOrder'].map(total_net).fillna(0.0)

    # ── Pandas → Spark ──────────────────────────────────────────────────────
    df_so  = spark.createDataFrame(so_pdf)
    df_soi = spark.createDataFrame(soi_pdf)

    # ── Persist to Unity Catalog Delta tables ───────────────────────────────
    (
        df_so.write.format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(_TARGET_SO)
    )
    (
        df_soi.write.format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(_TARGET_SOI)
    )

    print(f'[SalesOrder]     {df_so.count():,} rows → {_TARGET_SO}')
    print(f'[SalesOrderItem] {df_soi.count():,} rows → {_TARGET_SOI}')

    return {'SalesOrder': df_so, 'SalesOrderItem': df_soi}

In [ ]:

result = generate_sales_order_tables(reference_date='2023-01-01', months=42)
display(result['SalesOrder'])
display(result['SalesOrderItem'])